# TODO 6.4：SmoothQuant INT8 / FP8 能力验证

使用 NVIDIA ModelOpt 对真实模型执行 W8A8 SmoothQuant。FP8 只有在目标设备成功完成真实 FP8 矩阵运算后才记为 `pass`；仅存在 PyTorch FP8 dtype 不算通过。完整 INT8 状态写入 `banking_llm_project`，分析证据写入 `OptimizationSystem/out`。

In [1]:
%pip -q install -U "nvidia-modelopt[torch]" accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。请重启 Colab 会话；重启后跳过本单元。')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 48.0 MB/s eta 0:00:00
安装完成。请重启 Colab 会话；重启后跳过本单元。


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
CONFIG_PATH = PROJECT_DIR / 'configs/int8_fp8.json'
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/int8_fp8')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.int8_fp8 import (
    build_smoothquant_plan, collect_int8_fp8_evidence,
    run_smoothquant_plan, validate_int8_fp8_evidence,
)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU 运行时。'
assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plan = build_smoothquant_plan(config)
print('GPU:', torch.cuda.get_device_name(0))
print('校准样本:', len(assets.calibration_prompts))
print('计划:', plan.to_dict())

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
校准样本: 1024
计划: {'plan_id': 'smoothquant-int8-w8a8', 'method': 'smoothquant', 'weight_bits': 8, 'activation_bits': 8, 'alpha': 0.5, 'quantization_granularity': 'per_channel_weight_per_tensor_activation', 'backend_package': 'nvidia-modelopt'}


In [3]:
plan_result_dir = RESULT_DIR / plan.plan_id
plan_model_dir = MODEL_ROOT / plan.plan_id
manifest = plan_result_dir / 'quantized_model_manifest.json'
if manifest.is_file():
    print('跳过已完成导出:', manifest)
else:
    result = run_smoothquant_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=plan_result_dir,
        model_output_dir=plan_model_dir,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
        calibration_batch_size=int(config['calibration_batch_size']),
    )
    if result['status'] != 'pass':
        raise RuntimeError(result.get('detail', result))
    gc.collect(); torch.cuda.empty_cache()
print('INT8 manifest:', manifest)

/usr/local/lib/python3.13/dist-packages/modelopt/torch/__init__.py:55: UserWarning: transformers 5.18.0 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Registered <class 'transformers.models.qwen2.modeling_qwen2.Qwen2Attention'> to _QuantAttention for KV Cache quantization
Inserted 606 quantizers
Smoothed 168 modules


/usr/local/lib/python3.13/dist-packages/modelopt/torch/quantization/model_calib.py:1342: UserWarning: lm_head is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")


INT8 manifest: /content/drive/MyDrive/OptimizationSystem/out/int8_fp8/smoothquant-int8-w8a8/quantized_model_manifest.json


In [4]:
evidence = collect_int8_fp8_evidence(CONFIG_PATH, CALIBRATION_MANIFEST, RESULT_DIR)
evidence_path = RESULT_DIR / 'int8_fp8_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_int8_fp8_evidence(evidence_path)
payload = report.to_dict()
(RESULT_DIR / 'validation.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, ensure_ascii=False, indent=2))
statuses = {item['name']: item['status'] for item in payload['checks']}
assert statuses['smoothquant_real_execution'] == 'pass'
print('SmoothQuant INT8 真实导出：PASS')
print('NVIDIA FP8:', statuses['nvidia_fp8_capability'])
print('Ascend FP8:', statuses['ascend_fp8_capability'])

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/int8_fp8/int8_fp8_evidence.json",
  "complete": false,
  "checks": [
    {
      "name": "smoothquant_int8_config",
      "status": "pass",
      "detail": "SmoothQuant W8A8 configuration and calibration are recorded."
    },
    {
      "name": "smoothquant_export_contract",
      "status": "pass",
      "detail": "INT8 export and event-log names are fixed."
    },
    {
      "name": "smoothquant_backend_recorded",
      "status": "pass",
      "detail": "NVIDIA ModelOpt version is recorded."
    },
    {
      "name": "smoothquant_quantized_artifacts",
      "status": "pass",
      "detail": "SmoothQuant config, manifest, and event log exist."
    },
    {
      "name": "nvidia_fp8_capability",
      "status": "not_run",
      "detail": "A real FP8 matrix multiplication completed on NVIDIA CUDA."
    },
    {
      "name": "ascend_fp8_capability",
      "status": "not_run",
      "detail": "A real FP8 matrix multiplicati

In [5]:
import gc, json, sys
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BASELINE_REPORT = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_quality_report.json'
INT8_MANIFEST = RESULT_DIR / 'smoothquant-int8-w8a8/quantized_model_manifest.json'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.finqa_metrics import evaluate_numeric_answers
from src.quantization.int8_fp8 import (
    check_structured_json, evaluate_int8_outputs,
    load_smoothquant_service, validate_int8_evaluation,
)
if not INT8_MANIFEST.is_file():
    raise RuntimeError('没有真实 INT8 manifest；请先运行 int8_fp8_experiment_colab.ipynb。')
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
baseline_metrics = json.loads(BASELINE_REPORT.read_text(encoding='utf-8'))['metrics']
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, RESULT_DIR / 'int8_quality_service.jsonl')
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_comparison_service.jsonl')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Inserted 606 quantizers


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
predictions, references = [], []
batch_size = int(quality_config['batch_size'])
for start in range(0, len(assets.quality_dev), batch_size):
    rows = [assets.quality_dev[index] for index in range(start, min(start + batch_size, len(assets.quality_dev)))]
    requests = [InferenceRequest.from_finqa_row(row, request_id=f'int8-quality-{start + offset + 1:06d}') for offset, row in enumerate(rows)]
    results = int8_service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))
    predictions.extend(result['output_text'] for result in results)
    references.extend(str(row.get('answer') or row.get('qa', {}).get('answer')) for row in rows)
print('完整 dev 样本完成:', len(predictions))

Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 58.6 seconds
完整 dev 样本完成: 883


In [7]:
rows = [dict(row) for row in assets.quality_dev]
rows.sort(key=lambda row: len(InferenceRequest.from_finqa_row(row).render_prompt()), reverse=True)
long_results = []
for index, row in enumerate(rows[:3]):
    reference = str(row.get('answer') or row.get('qa', {}).get('answer'))
    request = InferenceRequest.from_finqa_row(row, request_id=f'long-{index + 1}')
    fp16_text = fp16_service.generate_one(request)['output_text']
    int8_text = int8_service.generate_one(request)['output_text']
    fp16_ok = evaluate_numeric_answers([fp16_text], [reference])['correct'] == 1
    int8_ok = evaluate_numeric_answers([int8_text], [reference])['correct'] == 1
    long_results.append({'index': index, 'input_characters': len(request.render_prompt()), 'baseline_passed': fp16_ok, 'passed': int8_ok})

structured_results = []
suffix = '\nReturn JSON with fields answer and unit only.'
for index, row in enumerate(rows[:10]):
    request = InferenceRequest.from_finqa_row(row, request_id=f'structured-{index + 1}')
    fp16_check = check_structured_json(fp16_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    int8_check = check_structured_json(int8_service.generate_one(request, prompt_suffix=suffix)['output_text'], ['answer', 'unit'])
    structured_results.append({'index': index, 'baseline_passed': fp16_check['passed'], **int8_check})